# Interactive browsing pairs of Image & landmarks

This simple visualization kit for https://www.kaggle.com/datasets/jirkaborovec/histology-cima-dataset

*The dataset consists of 2D histological microscopy tissue slices, stained with different stains, and landmarks denoting key points in each slice. The task is image registration - align all slices in a particular set of images (consecutive stain cuts) together, for instance to the initial image plane. The main challenges for these images are the following: very large image size, appearance differences, and lack of distinctive appearance objects. Our dataset contains 108 image pairs and manually placed landmarks for registration quality evaluation. A sample tissue of five stain cuts with highlighted landmarks is shown below.*

In [ ]:
import os, glob
import pandas
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/histology-cima-dataset"
SCALE = "scale-25pc"
# most images are in JPEG, just the 100% are PNG
IMAGE_EXTENSION = ".jpg"

### list tissue cases

In [ ]:
ls = glob.glob(os.path.join(DATASET_PATH, "*", SCALE, "*" + IMAGE_EXTENSION))
print(f"found {len(ls)} images with {SCALE}")
cases = sorted([os.path.basename(p) for p in glob.glob(os.path.join(DATASET_PATH, "*"))])
print(f"the tissue cases:\n {cases}")

## Show image and its landmarks

In [ ]:
def count_samples(case: str, scale: str = SCALE) -> int:
    ls = glob.glob(os.path.join(DATASET_PATH, case, scale, "*.jpg"))
    ls += glob.glob(os.path.join(DATASET_PATH, case, scale, "*.png"))
    return len(ls)

In [ ]:
def show_image_landmarks(
    case: str, scale: str = SCALE, sample_id: int = 0, img_ext: str = IMAGE_EXTENSION
):
    ls = glob.glob(os.path.join(DATASET_PATH, case, scale, "*" + img_ext))
    if sample_id >= len(ls):
        print(f"this case has only {len(ls)} samples and you asked for {sample_id}")
        return plt.figure()
    img_path = ls[sample_id]
    img = plt.imread(img_path)
    csv_path = img_path.replace(img_ext, ".csv")
    ldk = pandas.read_csv(csv_path, index_col=0)
    fig = plt.figure()
    fig.gca().imshow(img)
    fig.gca().plot(ldk['X'], ldk['Y'], 'x')
    return fig

In [ ]:
for i, case in enumerate(cases):
    i = i % count_samples(case)
    show_image_landmarks(case, sample_id=1).show()

## Interactive browsing

**NOTE:** to be able to play with it you need to copy and run this notebook

In [ ]:
from ipywidgets import interact, Dropdown, IntSlider

def interactive_show(cases: list, max_idx: int) -> None:
    interact(
        lambda case, i: show_image_landmarks(case, sample_id=i).show(),
        case=Dropdown(options=cases, description="Tissue"),
        i=IntSlider(min=0, max=max_idx, step=1, value=0),
    )

max_idx = max([count_samples(case) for case in cases])
print(f"highest nb of samples: {max_idx}")
interactive_show(cases, max_idx)